In [12]:
using DataDrivenDiffEq
using ModelingToolkit
using OrdinaryDiffEq
using DataDrivenSparse
using LinearAlgebra

# Create a test problem
function lorenz(u, p, t)
    x, y, z = u

    ẋ = 10.0 * (y - x)
    ẏ = x * (28.0 - z) - y
    ż = x * y - (8 / 3) * z
    return [ẋ, ẏ, ż]
end

u0 = [1.0; 0.0; 0.0]
tspan = (0.0, 100.0)
dt = 0.1
prob = ODEProblem(lorenz, u0, tspan)
sol = solve(prob, Tsit5(), saveat = dt)

## Start the automatic discovery
ddprob = DataDrivenProblem(sol)

@variables t x(t) y(t) z(t)
u = [x; y; z]
basis = Basis(polynomial_basis(u, 5), u, iv = t)
opt = STLSQ(exp10.(-5:0.1:-1))
ddsol = solve(ddprob, basis, opt, options = DataDrivenCommonOptions(digits = 1))
println(get_basis(ddsol))

AssertionError: AssertionError: The solution is not successful. Abort.

In [22]:
using GLMakie

# 1. Setup the "Live" data
xs = ys = range(-20, 20, length = 100)
t_observable = Observable(0.0) # This is our "Master Clock"

# 2. Create a "Lifted" surface
# This function automatically re-runs whenever t_observable changes
zs = lift(t_observable) do t
    [sin(x + t) + cos(y + t) for x in xs, y in ys]
end

# 3. Setup Figure
fig = Figure(size = (1200, 800))
ax = Axis3(fig[1, 1], title = "Wave Evolution", azimuth = 0.3)

# Plot using the 'zs' observable
GLMakie.surface!(ax, xs, ys, zs, colormap = :viridis)

# 4. Animation Settings
fps = 60
seconds = 5
total_frames = fps * seconds
# 360 degrees in radians is 2π
camera_angles = range(0, 2π, length = total_frames)
time_values = range(0, 10, length = total_frames) # 't' goes from 0 to 10

# 5. Record
record(fig, "evolving_wave.mp4", 1:total_frames; framerate = fps) do frame
    # Update the time (moves the wave)
    t_observable[] = time_values[frame]
    
    # Update the camera (rotates the view)
    ax.azimuth[] = 45

end

"evolving_wave.mp4"